# Day 5 — Evaluating AI Models

**Workshop:** Mathematical Foundations of Modern AI

Companion notebook to the Day 5 lecture notes. Four experiments, each producing one of the failure modes named in the chapter:

1. **Bias-variance decomposition** on polynomial regression --- watch the classical picture emerge.
2. **Calibration** of an MNIST classifier --- reliability diagram, ECE, and the temperature-scaling fix.
3. **Distribution shift** --- a CNN trained on standard MNIST evaluated on rotated test sets, accuracy vs angle.
4. **OOD detection** --- using softmax confidence to flag Fashion-MNIST inputs to a MNIST classifier.

Runs on CPU. ~5 minutes end-to-end.

---

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## Experiment 1 — Bias-variance decomposition

True function: $f(x) = \sin(2 \pi x)$. Each training set is 20 noisy points. Fit polynomials of degree 1, 3, 9, 15. Repeat over many random training sets.

For each candidate degree we compute:
- **Bias²** = squared difference between the average prediction and the true function
- **Variance** = average variance of the predictions across training sets
- **Total error** = bias² + variance + noise

In [ ]:
def true_function(x):
    return np.sin(2 * np.pi * x)

def sample_dataset(n=20, noise=0.2, rng=None):
    rng = rng or np.random.default_rng()
    x = rng.uniform(0, 1, n)
    y = true_function(x) + noise * rng.standard_normal(n)
    return x, y

degrees = [1, 3, 9, 15]
n_train_sets = 100
x_grid = np.linspace(0, 1, 200)
y_true = true_function(x_grid)

fig, axes = plt.subplots(1, len(degrees), figsize=(16, 3.5), sharey=True)
results = {}
for ax, d in zip(axes, degrees):
    preds = np.zeros((n_train_sets, len(x_grid)))
    for i in range(n_train_sets):
        x_tr, y_tr = sample_dataset(rng=np.random.default_rng(i))
        coef = np.polyfit(x_tr, y_tr, d)
        preds[i] = np.polyval(coef, x_grid)
        if i < 25:
            ax.plot(x_grid, preds[i], color="steelblue", alpha=0.15, lw=0.8)
    ax.plot(x_grid, y_true, "k-", lw=2, label="true f")
    ax.plot(x_grid, preds.mean(axis=0), "r--", lw=2, label="mean fit")
    ax.set_title(f"degree {d}")
    ax.set_ylim(-2, 2)
    ax.set_xlabel("x")
    if d == degrees[0]:
        ax.set_ylabel("y")
    ax.legend(fontsize=8)
    # Store for decomposition plot
    bias_sq = (preds.mean(axis=0) - y_true) ** 2
    variance = preds.var(axis=0)
    results[d] = (bias_sq.mean(), variance.mean())
plt.suptitle("Polynomial fits across 100 random training sets"); plt.tight_layout(); plt.show()

In [ ]:
# Bias-variance trade-off as a function of model complexity
degrees_sweep = list(range(1, 16))
biases, variances = [], []
for d in degrees_sweep:
    preds = np.zeros((n_train_sets, len(x_grid)))
    for i in range(n_train_sets):
        x_tr, y_tr = sample_dataset(rng=np.random.default_rng(i))
        coef = np.polyfit(x_tr, y_tr, d)
        preds[i] = np.polyval(coef, x_grid)
    biases.append(((preds.mean(axis=0) - y_true) ** 2).mean())
    variances.append(preds.var(axis=0).mean())

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(degrees_sweep, biases, "o-", label="bias²")
ax.plot(degrees_sweep, variances, "s-", label="variance")
ax.plot(degrees_sweep, np.array(biases) + np.array(variances), "^-", label="bias² + variance", lw=2)
ax.set_xlabel("Polynomial degree (model capacity)")
ax.set_ylabel("Error component")
ax.set_yscale("log")
ax.set_title("Bias-variance trade-off as model complexity grows")
ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

The classical U-shape: bias falls as you add capacity (model can fit the truth) and variance rises (model now fits the noise too). The sweet spot here is around degree 3 or 5 --- enough to fit a sine wave, not so flexible that random training-set noise dominates the fit.

---

## Experiment 2 — Calibration on MNIST

Train a small classifier, then check whether its 90%-confidence predictions are actually right 90% of the time. They will not be.

In [ ]:
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

transform = transforms.ToTensor()
full_train = datasets.MNIST(root=".", train=True, download=True, transform=transform)
test_ds    = datasets.MNIST(root=".", train=False, download=True, transform=transform)
train_ds, val_ds = random_split(full_train, [55000, 5000], generator=torch.Generator().manual_seed(0))
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=256)
test_loader  = DataLoader(test_ds, batch_size=256)

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.head = nn.Sequential(
            nn.Flatten(), nn.Linear(32*7*7, 64), nn.ReLU(),
            nn.Linear(64, 10)
        )
    def forward(self, x): return self.head(self.conv(x))

model = CNN().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()
for epoch in range(3):
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad(); loss_fn(model(x), y).backward(); opt.step()
    print(f"epoch {epoch+1} done")

In [ ]:
# Get logits and labels on the test set
def collect_logits(model, loader):
    model.eval()
    logits_all, labels_all = [], []
    with torch.no_grad():
        for x, y in loader:
            logits_all.append(model(x.to(device)).cpu())
            labels_all.append(y)
    return torch.cat(logits_all), torch.cat(labels_all)

def reliability(logits, labels, n_bins=15):
    probs = F.softmax(logits, dim=1)
    conf, pred = probs.max(dim=1)
    correct = (pred == labels).float()
    bins = np.linspace(0, 1, n_bins + 1)
    mean_conf, mean_acc, weight = [], [], []
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (conf > lo) & (conf <= hi)
        if mask.sum() > 0:
            mean_conf.append(conf[mask].mean().item())
            mean_acc.append(correct[mask].mean().item())
            weight.append(mask.sum().item() / len(conf))
    mean_conf, mean_acc, weight = map(np.array, (mean_conf, mean_acc, weight))
    ece = float(np.sum(weight * np.abs(mean_conf - mean_acc)))
    return mean_conf, mean_acc, weight, ece

logits_test, labels_test = collect_logits(model, test_loader)
mc, ma, w, ece = reliability(logits_test, labels_test)
print(f"Test accuracy:               {(logits_test.argmax(dim=1) == labels_test).float().mean():.4f}")
print(f"Expected Calibration Error:  {ece:.4f}")

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0, 1], [0, 1], "k--", label="perfect calibration")
ax.bar(mc, ma, width=1/16, alpha=0.6, edgecolor="black", label="observed accuracy")
ax.set_xlabel("mean predicted confidence (bin)")
ax.set_ylabel("observed accuracy (bin)")
ax.set_title(f"Reliability diagram before calibration (ECE = {ece:.3f})")
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Temperature scaling: fit a single scalar T on the validation set
logits_val, labels_val = collect_logits(model, val_loader)
T = torch.ones(1, requires_grad=True)
opt_T = torch.optim.LBFGS([T], lr=0.1, max_iter=50)

def step():
    opt_T.zero_grad()
    loss = F.cross_entropy(logits_val / T, labels_val)
    loss.backward()
    return loss
opt_T.step(step)
T_opt = T.item()
print(f"Optimal temperature on val: T = {T_opt:.3f}")

# Re-measure on the test set, scaled
mc2, ma2, w2, ece2 = reliability(logits_test / T_opt, labels_test)
print(f"ECE before calibration: {ece:.4f}")
print(f"ECE after calibration:  {ece2:.4f}")

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0, 1], [0, 1], "k--", label="perfect")
ax.bar(mc2, ma2, width=1/16, alpha=0.6, edgecolor="black", label="after temperature scaling")
ax.set_xlabel("mean predicted confidence (bin)")
ax.set_ylabel("observed accuracy (bin)")
ax.set_title(f"Reliability after temperature scaling (T={T_opt:.2f}, ECE={ece2:.3f})")
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.legend(); plt.tight_layout(); plt.show()

A single scalar, fit on a held-out set, restores calibration with no retraining. This is the simplest and most consistently effective post-hoc calibration method.

---

## Experiment 3 — Distribution shift via rotation

Take the same CNN and evaluate it on increasingly rotated test sets. The CNN learned translation equivariance (Day 4) but not rotation equivariance, so accuracy should fall as rotation grows.

In [ ]:
import torchvision.transforms.functional as TF

def evaluate_rotated(model, angle: float, batch_size: int = 256):
    model.eval()
    loader = DataLoader(test_ds, batch_size=batch_size)
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x_rot = TF.rotate(x, angle=angle)
            preds = model(x_rot.to(device)).argmax(dim=1).cpu()
            correct += (preds == y).sum().item()
            total += y.numel()
    return correct / total

angles = list(range(0, 91, 10))
accs = [evaluate_rotated(model, a) for a in angles]
for a, acc in zip(angles, accs):
    print(f"angle {a:3d}°: accuracy = {acc:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(angles, accs, "o-", lw=2)
ax.set_xlabel("Rotation angle (degrees)")
ax.set_ylabel("Test accuracy")
ax.set_ylim(0, 1.05)
ax.set_title("Accuracy of an MNIST-trained CNN on rotated test images")
ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

A ten-degree rotation already costs a few percentage points; by 45° the model is essentially guessing for several digits. The CNN's translation equivariance was an architectural choice, but rotation invariance was not. Models inherit the symmetries you build in, not the ones you wish they had.

Two ways to handle this in production:

1. **Data augmentation:** add rotated copies during training. The model learns rotation invariance from data instead of from architecture.
2. **Equivariant architectures:** networks designed to be rotation-equivariant (group-equivariant CNNs, $\mathrm{SE}(2)$-equivariant layers) get it built in.

---

## Experiment 4 — OOD detection with softmax confidence

Same CNN. Test on MNIST (in-distribution) and on Fashion-MNIST (out-of-distribution). For each input, take the maximum softmax probability across the 10 classes as a confidence score. A well-behaved model should be unconfident on OOD inputs.

In [ ]:
fashion_ds = datasets.FashionMNIST(root=".", train=False, download=True, transform=transform)
fashion_loader = DataLoader(fashion_ds, batch_size=256)

def confidences(model, loader):
    model.eval()
    cs = []
    with torch.no_grad():
        for x, _ in loader:
            probs = F.softmax(model(x.to(device)), dim=1)
            cs.append(probs.max(dim=1).values.cpu().numpy())
    return np.concatenate(cs)

conf_mnist   = confidences(model, test_loader)
conf_fashion = confidences(model, fashion_loader)
print(f"MNIST mean confidence:         {conf_mnist.mean():.3f}")
print(f"Fashion-MNIST mean confidence: {conf_fashion.mean():.3f}")
print(f"  (Lower would be better --- the model should be unsure on OOD inputs)")

In [ ]:
# AUROC of the confidence-as-OOD-detector
from sklearn.metrics import roc_auc_score
y_true = np.concatenate([np.ones_like(conf_mnist), np.zeros_like(conf_fashion)])   # 1 = in-dist
y_score = np.concatenate([conf_mnist, conf_fashion])
auroc = roc_auc_score(y_true, y_score)
print(f"AUROC of softmax-confidence OOD detector: {auroc:.3f}")
print(f"  (1.0 = perfect separation, 0.5 = random)")

fig, ax = plt.subplots(figsize=(8, 4.5))
bins = np.linspace(0, 1, 40)
ax.hist(conf_mnist,   bins=bins, alpha=0.6, label="MNIST (in-dist)", density=True)
ax.hist(conf_fashion, bins=bins, alpha=0.6, label="Fashion-MNIST (OOD)", density=True)
ax.set_xlabel("max softmax confidence")
ax.set_ylabel("density")
ax.set_title(f"Confidence distribution: in-dist vs OOD  (AUROC = {auroc:.3f})")
ax.legend(); plt.tight_layout(); plt.show()

Two things are notable:

1. The model is **uncomfortably confident on Fashion-MNIST.** Mean confidence is well above what it should be for inputs that contain no digits at all. This is the canonical OOD failure mode of modern softmax classifiers.
2. There is **still some separation** --- the AUROC is well above 0.5 --- but it is not enough for a production-grade OOD detector. More sophisticated methods (Mahalanobis distance, energy scores, OOD-aware training) close this gap considerably.

If your deployed model can receive inputs from a wider universe than your training set, building an explicit OOD detector is usually worth the engineering. Softmax confidence is the cheapest baseline; it is also rarely sufficient.

---

## What you have measured

Four classical failure modes, each diagnosed with a single concrete experiment:

- The bias-variance trade-off as a function of model capacity --- visible in the U-shape of total error.
- Calibration as a separate concern from accuracy --- diagnosed by the reliability diagram and ECE, repaired by temperature scaling.
- Distribution shift as the gap between test-set performance and deployment performance --- visible in the accuracy-vs-angle curve.
- OOD detection failure as the model's misplaced confidence on inputs from a different distribution --- visible in the confidence histograms.

Each of these is a way that a model with high test accuracy can still be the wrong thing to ship. Knowing they exist, and knowing how to measure them, is the difference between a model that survives contact with the real world and one that does not.

---

## Closing the workshop

Five days. Five chapters. The vocabulary you now have:

- **Day 1.** A neural network is affine maps composed with non-linearities, operating on vectors. Embeddings, autoencoders, latent space.
- **Day 2.** Training is gradient descent on a loss. The chain rule (backpropagation) makes it feasible for millions of parameters. Adam, learning curves, what the four failure shapes look like.
- **Day 3.** When the goal is generation, the loss becomes a likelihood and the model becomes a distribution to sample from. VAEs, diffusion, the unifying picture.
- **Day 4.** Architecture is an inductive bias. CNNs, GNNs, attention --- three different priors for three different kinds of data. Choosing the right one matters more than choosing the right hyperparameters.
- **Day 5.** Evaluation is the discipline that tells you whether all of the above is doing the right thing. Bias-variance, calibration, distribution shift, OOD detection, the ship/collect/redesign decision.

That is the workshop. You can read a deep-learning paper now and know what each piece is doing. You can talk to an ML team and ask the right questions. You can audit a deployed model and find the parts that should not be ignored. The mathematics underneath is not deep --- it is linear algebra, calculus, probability, and a lot of careful empirical work. The hard part was never the math. It was knowing where to apply it.

Thank you for the week.